In [12]:
import numpy as np, json, requests
OLLAMA = "http://10.42.0.247:11434/"
#GEN_MODEL = "qwen2.5:14b-instruct" # runs on cloudwright after Tuesday
#GEN_MODEL = "llama3.1:70b-instruct-q4_K_M"
GEN_MODEL = "gpt-oss:20b"
def ask(query):
    prompt = f"""{query}"""

    r = requests.post(f"{OLLAMA}/api/generate",
        json={"model": GEN_MODEL, "prompt": prompt, "stream": False}, timeout=1000)
    print( r)
    print(r.json())
    resp = r.json()["response"].strip()
    return resp


In [16]:
tools_description = """

**Command Interface**
you and i will communicate in json blocks

you will receive a json block like:

{
 "format" : "json_plain",
 "query" : "an optional query from the user, if this is present, address it",
 "tool_call_result" : "optional result of the last tool call, it will start with one line that holds the issued tool call text, followed by the results",
 "thread_summary" : "current summary of this working session"
}

for any response return only a JSON block like the following, no preamble or conclusion

{
 "response" : "optional: any text response to the user, this part will not be executed, but the user will have to reply if this is present, if you dont need input from the user on this loop omit this key",
 "tool_call" : "a literal tool call from the following exposed list, this part will execute, you will receive the results on the next call, only output one tool call as a plain string exactly as it would be entered on the commandline",
 "thread_summary" : "a progressive summary of this conversation that includes the information from the thread_summary you were sent, try to keep it under 20k"
}

**exposed tools**

* `ls <options>`: List files in the local working area
* `cat <filename>`: Display the contents of a file
* `pwd` : print the working directory

**Using the Tool**

* the working dir will not change, issue all commands from the inital working dir
* Issue a command using the following syntax: `command arguments...`
* Use quotes to enclose arguments with spaces (e.g., `cat "example file.txt"`)
* if the result of a tool call is unexpected, consult the user

**Example tool_call Input**

* `ls`
* `ls <some options>`
* `cat example.txt`

Please respond with a json_plain response the starts with '{' and ends with '}', no formatting or unrequired newlines, 
without any additional text or narrative explanation.
make sure to include context in the thread_summary field

"""

start_loop = f"""

hi there, im doing some experiments giving you control over the command line, please dont hurt me, im actually not that bad of a person

the first thing we will do is see if you can use the following tool set to locate the "projects" directory in the working dir

then locate the cae directory and use the cat command to read the README.md file then summarize this file

if you have questions, ask them per the following interface

{tools_description}
"""

In [17]:
import json
import subprocess
import pathlib

def execute_command(command):
    process = subprocess.run(command, shell=True, capture_output=True)
    return process.stdout.decode('utf-8')

# Main loop
def execute(command):
    if command.startswith('ls'):
        output = execute_command(command)
    elif command.startswith('cat'):
        output = execute_command(command)
    elif command == 'pwd':
        output = execute_command(command)
    else:
        output = 'Unknown command'
        return output, "error"
    return output, None


In [ ]:
from IPython.display import clear_output
    
thread = {
 "format" : "json_plain",
 "query" : start_loop,
 "thread_summary" : ""
}

for i in range(5):
    
    clear_output()
    print(thread)
    reply = ask(thread)
    query = ""
    try:
        reply = json.loads(reply)
        
        next_call = {"format" : "json_plain", "thread_summary" : reply["thread_summary"]}
        
        if 'response' in reply:
            print(GEN_MODEL, reply['response'])
            print("wants to run: ", reply['tool_call'])
            query = input('LLM: $ ')
        if 'tool_call' in reply:
            print(reply['tool_call'])
            output, err = execute(reply['tool_call'])
            print(err)
            next_call["tool_call_result"] = f"""
{reply['tool_call']}
{output}
{err if err is not None else ""}"""
        
        next_call["query"] = f""" 
{query}

{tools_description}"""
        thread = json.dumps(next_call)
    except Exception as e:
        print(e)
        break

    

{"format": "json_plain", "thread_summary": "Searching for projects directory. Listing contents of projects directory.", "tool_call_result": "\nls projects\ncae\nmind-body-engineer\nmind-body-engineer-l1\nqwen35\n\n", "query": " \n\n\n\n\n**Command Interface**\nyou and i will communicate in json blocks\n\nyou will receive a json block like:\n\n{\n \"format\" : \"json_plain\",\n \"query\" : \"an optional query from the user, if this is present, address it\",\n \"tool_call_result\" : \"optional result of the last tool call, it will start with one line that holds the issued tool call text, followed by the results\",\n \"thread_summary\" : \"current summary of this working session\"\n}\n\nfor any response return only a JSON block like the following, no preamble or conclusion\n\n{\n \"response\" : \"optional: any text response to the user, this part will not be executed, but the user will have to reply if this is present, if you dont need input from the user on this loop omit this key\",\n \

In [11]:
reply


''